In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/hillstrom.db')
df = pd.read_sql("SELECT * FROM experiment", conn)
df['treatment_flag'] = (df['treatment'] == 'Womens E-Mail').astype(int)

print('='*50)
print(df.shape)
print('='*50)
print(df[['channel', 'zip_code']].nunique())
print('='*50)
print(df['zip_code'].unique())
print('='*50)

# history_segment is dropped, it's just history binned into ranges,
# so keeping both would add redundant, collinear columns for no new information.
features = ['recency', 'history', 'mens', 'womens', 'newbie', 'channel', 'zip_code']

print(df[features].dtypes)
print('='*50)
print(df[features].isna().sum())
print('='*50)

(42693, 13)
channel     3
zip_code    3
dtype: int64
<ArrowStringArray>
['Surburban', 'Rural', 'Urban']
Length: 3, dtype: str
recency       int64
history     float64
mens          int64
womens        int64
newbie        int64
channel         str
zip_code        str
dtype: object
recency     0
history     0
mens        0
womens      0
newbie      0
channel     0
zip_code    0
dtype: int64


In [2]:
'''Encoded on the full dataset, not train-only, one-hot encoding of fixed categorical
values (channel, zip_code) doesn't depend on the outcome, unlike target encoding or
scaling, so this doesn't leak target information. Doing it before the split also
guarantees train and test see the same set of dummy columns.'''
X = pd.get_dummies(df[features], columns=['channel', 'zip_code'], drop_first=False)

print('='*50)
print(X.shape)
print('='*50)
print(X.columns.tolist())
print('='*50)

from sklearn.model_selection import train_test_split

y_visit = df['visit']
treatment_binary = df['treatment_flag']
spend = df['spend']

X_train, X_test, y_train, y_test, t_train, t_test, spend_train, spend_test = train_test_split(
    X, y_visit, treatment_binary, spend,
    test_size=0.3, stratify=treatment_binary, random_state=42
)

print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print('='*50)
print("\nTreatment balance, train:")
print('='*50)
print(t_train.value_counts(normalize=True).round(3))
print('='*50)
print("\nTreatment balance, test:")
print('='*50)
print(t_test.value_counts(normalize=True).round(3))
print('='*50)

(42693, 11)
['recency', 'history', 'mens', 'womens', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web', 'zip_code_Rural', 'zip_code_Surburban', 'zip_code_Urban']
Train: 29885 rows, Test: 12808 rows

Treatment balance, train:
treatment_flag
1    0.501
0    0.499
Name: proportion, dtype: float64

Treatment balance, test:
treatment_flag
1    0.501
0    0.499
Name: proportion, dtype: float64


In [3]:
train_df = X_train.copy()
train_df['visit'] = y_train
train_df['treatment_flag'] = t_train
train_df['spend'] = spend_train

test_df = X_test.copy()
test_df['visit'] = y_test
test_df['treatment_flag'] = t_test
test_df['spend'] = spend_test

train_df.to_parquet('../data/train.parquet', index=False)
test_df.to_parquet('../data/test.parquet', index=False)

print('='*50)
print(f"train.parquet: {train_df.shape}")
print('='*50)
print(f"test.parquet:  {test_df.shape}")
print('='*50)

train.parquet: (29885, 14)
test.parquet:  (12808, 14)


**Feature set locked.** Final `X` has 11 columns: `recency`, `history`, `mens`, `womens`, `newbie`, and one-hot encodings of `channel` (3 categories) and `zip_code` (3 categories, including the dataset's own "Surburban" spelling). `history_segment` was dropped as a redundant binning of `history`. Split is 70/30 stratified by `treatment_flag`, `random_state=42`, saved as `data/train.parquet` and `data/test.parquet` so every later notebook loads the identical split. Encoding was done on the full dataset before splitting, safe here since one-hot encoding of fixed categories carries no target information, unlike scaling or target encoding.